In [ ]:
#Task 1: Top 3 Customers per City Calculate total spend per customer, then use window function to rank
customers within each city. Output: city, customer_id, total_spend, rank


from pyspark.sql.functions import sum as _sum
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

# total spend per customer per city
cust_spend = fact_orders.groupBy("customer_id", "customer_city") \
    .agg(_sum("payment_value").alias("total_spend"))

# window for ranking
window_spec = Window.partitionBy("customer_city").orderBy(col("total_spend").desc())

# ranking
top_customers = cust_spend.withColumn("rank", dense_rank().over(window_spec)) \
    .filter(col("rank") <= 3)

display(top_customers.select(
    col("customer_city").alias("city"),
    "customer_id",
    "total_spend",
    "rank"
))



In [ ]:
#Task 2: Running Total of Sales Calculate daily sales and then cumulative (running) total using window
function. Output: date, daily_sales, running_total

from pyspark.sql.functions import to_date

daily_sales = fact_orders.withColumn("date", to_date("order_purchase_timestamp")) \
    .groupBy("date") \
    .agg(_sum("payment_value").alias("daily_sales"))

window_spec = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

running_sales = daily_sales.withColumn(
    "running_total",
    _sum("daily_sales").over(window_spec)
)

display(running_sales)



In [ ]:
#Task 3: Top Products per Category Aggregate sales per product, join with category, then rank using
DENSE_RANK. Output: category, product_id, total_sales, rank


# total sales per product
product_sales = fact_orders.groupBy("product_id", "product_category_name") \
    .agg(_sum("payment_value").alias("total_sales"))

window_spec = Window.partitionBy("product_category_name") \
    .orderBy(col("total_sales").desc())

ranked_products = product_sales.withColumn(
    "rank",
    dense_rank().over(window_spec)
)

display(ranked_products.select(
    col("product_category_name").alias("category"),
    "product_id",
    "total_sales",
    "rank"
))



In [ ]:
#Task 4: Customer Lifetime Value Calculate total spend per customer across all orders. Output:
customer_id, total_spend

clv = fact_orders.groupBy("customer_id") \
    .agg(_sum("payment_value").alias("total_spend"))

display(clv)



In [ ]:
#Task 5: Customer Segmentation Apply logic: total_spend > 10000 → Gold 5000–10000 → Silver <5000
→ Bronze Add a new column 'segment'. Then group by segment to count customers. Output:
customer_id, total_spend, segment(changed the values to 50 and 100 instead of 5000 and 10000)

from pyspark.sql.functions import when

segmentation = clv.withColumn(
    "segment",
    when(col("total_spend") > 100, "Gold")
    .when((col("total_spend") >= 50) & (col("total_spend") <= 100), "Silver")
    .otherwise("Bronze")
)

# count customers per segment
segment_count = segmentation.groupBy("segment").count()

display(segmentation)
display(segment_count)
#segment count
Bronze	16610
Gold	53532
Silver	28523



In [ ]:
#Task 6: Final Reporting Table Combine results into a single dataset containing: customer_id, city,
total_spend, segment, total_orders


# total orders per customer
orders_count = fact_orders.groupBy("customer_id") \
    .count() \
    .withColumnRenamed("count", "total_orders")

# customer city
customer_city = fact_orders.select("customer_id", "customer_city").distinct()

# final join
final_report = segmentation \
    .join(customer_city, "customer_id", "left") \
    .join(orders_count, "customer_id", "left")

display(final_report.select(
    "customer_id",
    col("customer_city").alias("city"),
    "total_spend",
    "segment",
    "total_orders"
))